# Quantum Oracle Sketching — Full Benchmark Suite
**Tommaso R. Marena (2026)**

Reproduces all benchmark figures from the paper:
- Section 4: Core QOS scaling laws (flat vector, general vector, Boolean oracle, matrix element, matrix row-index)
- Section 5.1: Adaptive sparse oracle (N/K improvement)
- Section 5.2: Hierarchical sketching (Q^{2−1/k} barrier)
- Section 5.3: Variational warmstart
- Section 5.4: Interferometric shadow
- Appendix: Noise model, k-Forrelation, kernel shadow, non-IID scaling

**Recommended runtime:** Google Colab Pro+ with A100 GPU (~5–8 hrs full, ~1–2 hrs reduced).
Set `FAST_MODE = True` below for a quick reduced sweep (~15–30 min on A100).

## ⚠️ Compute Requirements

| Mode | Dims | Max samples | Reps | A100 time | T4 time | CPU time |
|---|---|---|---|---|---|---|
| `FAST_MODE=True` | 64, 256, 1024 | 1M | 5 | ~20 min | ~1.5 hr | ~6 hr |
| `FAST_MODE=False` (paper quality) | 100, 1000, 10000 | 100M | 10 | ~6 hr | ~15 hr | ~50 hr |

> The bottleneck is `dim=10000` + `M=100M` in `q_state_sketch` (QSVT path) and
> `benchmark_random_sparse_matrix_row_index` which loops over rows in pure Python.
> All other benchmarks are fully JAX-vmapped and GPU-accelerated.

In [ ]:
# Install repo (Colab only — skip if running locally)
import subprocess, sys
result = subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git#egg=qos[dev,noise,kernel]'],
    capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else 'Install complete')
print(result.stderr[-1000:] if result.returncode != 0 else '')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  CONFIGURATION — edit here
# ─────────────────────────────────────────────────────────────────────────────

FAST_MODE   = True   # True = quick (~20 min A100), False = paper quality (~6 hr A100)
SAVE_FIGS   = True   # Save PDFs to /results/
SHOW_FIGS   = True   # Display inline
SEED        = 42
OUTPUT_DIR  = './results'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

if FAST_MODE:
    DIM_LIST          = [64, 256, 1024]
    SAMPLES_LIST      = [10_000, 100_000, 1_000_000]
    REPETITION        = 5
    EXT_DIM           = 256      # dimension for extension benchmarks
    EXT_TRIALS        = 3
else:
    DIM_LIST          = [100, 1000, 10000]
    SAMPLES_LIST      = [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
    REPETITION        = 10
    EXT_DIM           = 256
    EXT_TRIALS        = 5

print(f'Mode: {"FAST" if FAST_MODE else "PAPER QUALITY"}')
print(f'Dims: {DIM_LIST}')
print(f'Sample counts: {SAMPLES_LIST}')
print(f'Repetitions: {REPETITION}')

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from tqdm.notebook import tqdm
import time, json

# Verify JAX device
print('JAX devices:', jax.devices())
print('JAX version:', jax.__version__)

from qos.experiments.benchmark import (
    run_benchmark_sweep,
    benchmark_random_flat_vector,
    benchmark_random_vector,
    benchmark_random_boolean_function,
    benchmark_random_sparse_matrix_element,
    benchmark_random_sparse_matrix_row_index,
    fit_sample_complexity,
    plot_benchmark_results,
)

key = random.PRNGKey(SEED)
print('Setup complete.')

## Section 4: Core QOS Scaling Laws

### 4.1 Flat Vector State Sketching

In [ ]:
key, subkey = random.split(key)
t0 = time.time()
flat_results = run_benchmark_sweep(
    subkey,
    benchmark_random_flat_vector,
    dim_list=DIM_LIST,
    unit_num_samples_list=SAMPLES_LIST,
    repetition=REPETITION,
)
flat_fit = fit_sample_complexity(flat_results)
print(f'Elapsed: {time.time()-t0:.1f}s')

plot_benchmark_results(
    flat_results, title='Flat vector state sketching',
    dim_list=DIM_LIST, fit=flat_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_flat_vector.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS,
)

### 4.2 General Vector State Sketching (QSVT arcsin path)

In [ ]:
key, subkey = random.split(key)
t0 = time.time()
vector_results = run_benchmark_sweep(
    subkey,
    benchmark_random_vector,
    dim_list=DIM_LIST,
    unit_num_samples_list=SAMPLES_LIST,
    repetition=REPETITION,
)
vector_fit = fit_sample_complexity(vector_results)
print(f'Elapsed: {time.time()-t0:.1f}s')

plot_benchmark_results(
    vector_results, title='General vector state sketching',
    dim_list=DIM_LIST, fit=vector_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_general_vector.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS,
)

### 4.3 Boolean Phase-Oracle Sketching

In [ ]:
key, subkey = random.split(key)
t0 = time.time()
boolean_results = run_benchmark_sweep(
    subkey,
    benchmark_random_boolean_function,
    dim_list=DIM_LIST,
    unit_num_samples_list=SAMPLES_LIST,
    repetition=REPETITION,
)
boolean_fit = fit_sample_complexity(boolean_results)
print(f'Elapsed: {time.time()-t0:.1f}s')

plot_benchmark_results(
    boolean_results, title='Boolean oracle sketching',
    dim_list=DIM_LIST, fit=boolean_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_boolean_function.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS,
)

### 4.4 Sparse Matrix Element Oracle

In [ ]:
key, subkey = random.split(key)
t0 = time.time()
element_results = run_benchmark_sweep(
    subkey,
    benchmark_random_sparse_matrix_element,
    dim_list=DIM_LIST,
    unit_num_samples_list=SAMPLES_LIST,
    repetition=REPETITION,
    matrix_dim=max(DIM_LIST),
)
element_fit = fit_sample_complexity(element_results)
print(f'Elapsed: {time.time()-t0:.1f}s')

plot_benchmark_results(
    element_results, title='Sparse matrix element oracle',
    dim_list=DIM_LIST, fit=element_fit,
    dim_label='nnz', dim_fit_label='nnz',
    save_path=f'{OUTPUT_DIR}/benchmark_matrix_element.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS,
)

### 4.5 Sparse Matrix Row-Index Oracle

In [ ]:
key, subkey = random.split(key)
t0 = time.time()
row_index_results = run_benchmark_sweep(
    subkey,
    benchmark_random_sparse_matrix_row_index,
    dim_list=DIM_LIST,
    unit_num_samples_list=SAMPLES_LIST,
    repetition=REPETITION,
)
row_index_fit = fit_sample_complexity(row_index_results)
print(f'Elapsed: {time.time()-t0:.1f}s')

plot_benchmark_results(
    row_index_results, title='Sparse matrix row-index oracle',
    dim_list=DIM_LIST, fit=row_index_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_matrix_row_index.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS,
)

## Section 5.1: Adaptive Sparse Oracle (N/K Improvement)

In [ ]:
from qos.core.oracle_sketch import (
    q_oracle_sketch_boolean,
    q_oracle_sketch_boolean_adaptive,
)

N      = EXT_DIM
K_vals = [max(1, N // k) for k in [1, 4, 16, 64]]   # sparsity levels
M_vals = [1_000, 5_000, 10_000, 50_000, 100_000]

uniform_errors, adaptive_errors = {}, {}

for K in tqdm(K_vals, desc='Sparsity K'):
    ue_list, ae_list = [], []
    key, subkey = random.split(key)
    f = jnp.zeros(N).at[:K].set(1).astype(jnp.int32)
    target = jnp.exp(1j * jnp.pi * f)
    for M in M_vals:
        u_diag, _ = q_oracle_sketch_boolean(f, M)
        a_diag, _, _ = q_oracle_sketch_boolean_adaptive(f, M, pilot_frac=0.1,
                            key=random.PRNGKey(SEED))
        ue_list.append(float(jnp.max(jnp.abs(u_diag - target))))
        ae_list.append(float(jnp.max(jnp.abs(a_diag - target))))
    uniform_errors[K] = ue_list
    adaptive_errors[K] = ae_list

# Plot
fig, axes = plt.subplots(1, len(K_vals), figsize=(4*len(K_vals), 3), sharey=True)
cmap = plt.get_cmap('Oranges')
for ax, K in zip(axes, K_vals):
    ax.plot(M_vals, uniform_errors[K], 'o-', color=cmap(0.5), label='Uniform (Zhao et al.)')
    ax.plot(M_vals, adaptive_errors[K], 's--', color=cmap(0.85), label='Adaptive (Marena 2026)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title(f'K={K}, N/K={N//max(K,1)}×')
    ax.set_xlabel('Samples M')
    ax.grid(True, which='both', alpha=0.3)
axes[0].set_ylabel('Max error ε')
axes[0].legend()
plt.suptitle(f'Adaptive Boolean Oracle: N={N}', y=1.02)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/adaptive_oracle.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

## Section 5.2: Hierarchical Sketching (Q^{2−1/k} Barrier)

In [ ]:
from qos.theory import HierarchicalOracleSketch
from qos.theory.hierarchical_sketch import compute_hierarchical_sample_complexity

N = EXT_DIM
Q_vals = [4, 8, 16, 32]
k_vals = [1, 2, 3, 4]       # hierarchy levels

zhao_samples, hier_samples = {}, {}

for Q in tqdm(Q_vals, desc='Query budget Q'):
    zhao_samples[Q] = N * Q**2
    hier_samples[Q] = {}
    for k in k_vals:
        stats = compute_hierarchical_sample_complexity(N=N, Q=Q, num_levels=k)
        hier_samples[Q][k] = stats['total_samples']

fig, ax = plt.subplots(figsize=(5, 3.5))
cmap = plt.get_cmap('Oranges')
ax.plot(Q_vals, [zhao_samples[Q] for Q in Q_vals], 'k--o', label='Zhao et al. O(NQ²)', lw=2)
for i, k in enumerate(k_vals):
    ax.plot(Q_vals, [hier_samples[Q][k] for Q in Q_vals],
            's-', color=cmap(0.35 + 0.2*i), label=f'Hierarchical k={k}')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('Query budget Q'); ax.set_ylabel('Sample complexity M')
ax.set_title(f'Hierarchical vs Zhao et al. (N={N})')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/hierarchical_sketch.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

## Section 5.3: Variational Warmstart

In [ ]:
from qos.theory import VariationalWarmstart

N = EXT_DIM
KF_vals = [max(1, N // k) for k in [1, 4, 16]]   # Fourier sparsity levels
M_train = 5000

vw_errors, baseline_errors = {}, {}

for KF in tqdm(KF_vals, desc='Fourier sparsity KF'):
    key, subkey = random.split(key)
    # Sparse Fourier truth table
    f = jnp.zeros(N).at[:KF].set(1).astype(jnp.int32)
    target = jnp.exp(1j * jnp.pi * f)
    
    vw = VariationalWarmstart(
        f, num_fourier_modes=min(KF, 32),
        learning_rate=0.02, num_steps=100
    )
    vw.fit(unit_num_samples=M_train)
    vw_diag = vw.predict()
    
    baseline_diag, _ = q_oracle_sketch_boolean(f, M_train)
    
    vw_errors[KF] = float(jnp.max(jnp.abs(vw_diag - target)))
    baseline_errors[KF] = float(jnp.max(jnp.abs(baseline_diag - target)))
    print(f'KF={KF}: baseline={baseline_errors[KF]:.4f}, variational={vw_errors[KF]:.4f}')

fig, ax = plt.subplots(figsize=(4.5, 3))
x = range(len(KF_vals))
ax.bar([xi - 0.2 for xi in x], [baseline_errors[KF] for KF in KF_vals],
       0.35, label='Uniform baseline', color='#888888')
ax.bar([xi + 0.2 for xi in x], [vw_errors[KF] for KF in KF_vals],
       0.35, label='Variational warmstart', color=plt.get_cmap('Oranges')(0.75))
ax.set_xticks(list(x)); ax.set_xticklabels([f'KF={kf}' for kf in KF_vals])
ax.set_ylabel('Max oracle error ε'); ax.set_yscale('log')
ax.set_title(f'Variational Warmstart vs Baseline (N={N}, M={M_train})')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/variational_warmstart.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

## Section 5.4: Interferometric Classical Shadow

In [ ]:
from qos.experiments.noise_benchmark import run_noise_benchmark
from qos.theory import InterferometricClassicalShadow
import numpy as np

key, subkey = random.split(key)
N = EXT_DIM
num_shadows_list = [50, 100, 200, 500, 1000]

shadow_errors = []
# Random unit weight state
weight_state = random.normal(subkey, (N,))
weight_state = weight_state / jnp.linalg.norm(weight_state)

# Random test vectors
key, subkey = random.split(key)
test_vecs = random.normal(subkey, (20, N))
test_vecs = test_vecs / jnp.linalg.norm(test_vecs, axis=1, keepdims=True)

for ns in tqdm(num_shadows_list, desc='Shadow count'):
    shadow = InterferometricClassicalShadow(weight_state, num_shadows=ns)
    shadow.build_shadow()
    preds = shadow.predict(test_vecs)
    # Ground truth: <weight|test>
    gt = jnp.einsum('d,td->t', weight_state, test_vecs)
    err = float(jnp.mean(jnp.abs(preds[:, 0] - jnp.real(gt))))
    shadow_errors.append(err)
    print(f'T={ns}: mean error = {err:.4f}')

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(num_shadows_list, shadow_errors, 'o-', color=plt.get_cmap('Oranges')(0.75))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Number of shadows T'); ax.set_ylabel('Mean prediction error')
ax.set_title('Interferometric Shadow Prediction Error')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/interferometric_shadow.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

## Appendix A: Depolarizing Noise Model

In [ ]:
from qos.experiments.noise_benchmark import run_noise_benchmark

noise_results = run_noise_benchmark(
    dim=EXT_DIM,
    num_trials=EXT_TRIALS,
    output_dir=OUTPUT_DIR,
)
print('Noise benchmark complete. Results:', list(noise_results.keys()))

## Appendix B: k-Forrelation Lower Bound Benchmark

In [ ]:
from qos.experiments.forrelation_benchmark import run_forrelation_benchmark

forrelation_results = run_forrelation_benchmark(
    dim=EXT_DIM,
    num_trials=EXT_TRIALS,
    output_dir=OUTPUT_DIR,
)
print('Forrelation benchmark complete.')

## Appendix C: Kernel Shadow

In [ ]:
from qos.experiments.kernel_benchmark import run_kernel_benchmark

kernel_results = run_kernel_benchmark(
    dim=EXT_DIM,
    num_trials=EXT_TRIALS,
    output_dir=OUTPUT_DIR,
)
print('Kernel benchmark complete.')

## Appendix D: Non-IID Scaling

In [ ]:
from qos.experiments.non_iid_scaling import run_non_iid_scaling

noniid_results = run_non_iid_scaling(
    dim=EXT_DIM,
    num_trials=EXT_TRIALS,
    output_dir=OUTPUT_DIR,
)
print('Non-IID benchmark complete.')

## Summary: Improvement Factors

In [ ]:
print('=' * 70)
print(f'BENCHMARK SUMMARY  (N={EXT_DIM}, FAST_MODE={FAST_MODE})')
print('=' * 70)
print(f'{"Contribution":<35} {"Zhao et al.":<20} {"Marena 2026":<20}')
print('-' * 70)

# Adaptive: compare errors at same M
M_compare = M_vals[len(M_vals)//2]
K_test = max(1, EXT_DIM // 16)
idx = M_vals.index(M_compare)
u_err = uniform_errors[K_test][idx]
a_err = adaptive_errors[K_test][idx]
print(f'{"Adaptive oracle (K=N/16)":<35} {u_err:.4f}{" (err)":<14} {a_err:.4f} (err)')
print(f'{"":<35} {"O(N·Q²)":<20} {"O(K·Q²) = N/K×":<20}')
print()

# Hierarchical: sample count at Q=16, k=2
if 16 in Q_vals and 2 in k_vals:
    z_s = zhao_samples[16]
    h_s = hier_samples[16][2]
    print(f'{"Hierarchical (Q=16, k=2)":<35} {z_s:<20,} {h_s:<20,}')
    print(f'{"":<35} {"O(NQ²)":<20} {"O(NQ^(2-1/k))":<20}')
    print()

print('Results saved to:', OUTPUT_DIR)
import glob
for f in sorted(glob.glob(f'{OUTPUT_DIR}/*.pdf')):
    print(f'  {f}')